# 05 - APIs and Networking: a Minimal Request/Response Round Trip

Read `../concept.md` first. This notebook starts a tiny HTTP server with **Flask** (the most common Python web framework, and the one the frontend unit's backend examples will likely build on), makes real requests against it with the `requests` library, and shuts it down again -- all in this one notebook process, talking over `localhost`.

Two endpoints:

- `GET /status` -- returns the robot's current mock status as JSON.
- `POST /autonomous` -- accepts a JSON body selecting a new autonomous routine, and updates that status.

## Imports and shared state

`robot_status` is the "database" this whole demo reads from and writes to -- just a plain dict, standing in for whatever real state a robot status service would track.

In [1]:
import logging
import threading

from flask import Flask, jsonify, request
from werkzeug.serving import make_server
import requests

# Silence Flask/Werkzeug's default per-request access log -- we print our
# own, more readable lines from inside each route below instead.
logging.getLogger("werkzeug").setLevel(logging.ERROR)

robot_status = {
    "batteryVoltage": 12.6,
    "selectedRoutine": "doNothing",
    "sensorsHealthy": True,
}

/Users/nickmelamed/miniforge3/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


## The server

A Flask app is built from `@app.route(...)`-decorated functions -- one per endpoint -- instead of checking `self.path` by hand. Each route reads or writes `robot_status`, then returns a value Flask converts into a JSON response automatically (`jsonify`). A route that doesn't match anything falls through to the `404` handler below, exactly like a real server returns for an endpoint that doesn't exist.

In [2]:
app = Flask(__name__)


@app.route("/status", methods=["GET"])
def get_status():
    print("[server] GET /status -> 200")
    return jsonify(robot_status)


@app.route("/autonomous", methods=["POST"])
def post_autonomous():
    routine = request.get_json()["routine"]
    robot_status["selectedRoutine"] = routine
    print(f"[server] POST /autonomous -> 200 (selectedRoutine={routine!r})")
    return jsonify(robot_status)


@app.errorhandler(404)
def not_found(error):
    print(f"[server] {request.method} {request.path} -> 404")
    return jsonify({"error": f"no such route: {request.path}"}), 404

## Starting the server

`app.run()` is Flask's usual way to start a server, but it blocks and doesn't give you a clean way to stop it from elsewhere in a notebook. `werkzeug.serving.make_server` (the WSGI server Flask uses underneath `app.run()` anyway) gives back a server object with `serve_forever()`/`shutdown()`, so it can run in a background thread here -- leaving this notebook's main thread free to act as the *client* and make requests against it, in the same process. Binding to port `0` asks the OS for any free port, so this cell doesn't need to guess one that's open.

In [3]:
server = make_server("localhost", 0, app)
server_port = server.server_port

server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()

base_url = f"http://localhost:{server_port}"
print(f"server started at {base_url}")

server started at http://localhost:59828


## Making requests

This is the client half -- the same kind of call a dashboard's frontend code would make against a real backend. First, a `GET /status`.

Equivalent from a terminal:
```text
$ curl http://localhost:PORT/status
```

In [4]:
response = requests.get(f"{base_url}/status")
print("status code:", response.status_code)
print("body:       ", response.json())

status code: 200
body:        {'batteryVoltage': 12.6, 'selectedRoutine': 'doNothing', 'sensorsHealthy': True}


Now a `POST /autonomous`, selecting a new routine, with a JSON body in the request itself.

Equivalent from a terminal:
```text
$ curl -X POST http://localhost:PORT/autonomous -H "Content-Type: application/json" -d '{"routine": "leftStartTwoPiece"}'
```

In [5]:
response = requests.post(f"{base_url}/autonomous", json={"routine": "leftStartTwoPiece"})
print("status code:", response.status_code)
print("body:       ", response.json())

status code: 200
body:        {'batteryVoltage': 12.6, 'selectedRoutine': 'leftStartTwoPiece', 'sensorsHealthy': True}


Requesting `/status` again shows the change actually took effect on the server -- this is the same `robot_status` dict on both requests, updated by the `POST` above.

In [6]:
response = requests.get(f"{base_url}/status")
print("status code:", response.status_code)
print("body:       ", response.json())

status code: 200
body:        {'batteryVoltage': 12.6, 'selectedRoutine': 'leftStartTwoPiece', 'sensorsHealthy': True}


## A 404, for a route that doesn't exist

Same server, same client, just asking for something that was never defined.

In [7]:
response = requests.get(f"{base_url}/nonexistent")
print("status code:", response.status_code)
print("body:       ", response.json())

status code: 404
body:        {'error': 'no such route: /nonexistent'}


## Shutting the server down

Cleaning up after ourselves -- in a real deployment the server would just keep running, but this notebook shouldn't leave a background thread and an open port behind once you're done with it.

In [8]:
server.shutdown()
server.server_close()
print("server stopped")

server stopped


## Try It Yourself

No solutions are provided -- these are meant to be worked through on your own or with a mentor or another student.

1. Add a third endpoint, `GET /battery`, that returns just the `batteryVoltage` field on its own, not the whole status dict.
2. Make `POST /autonomous` return a `400` status code (with an error message body) if the request body is missing the `routine` key, instead of letting it crash.
3. Time ten back-to-back `GET /status` calls with `time.time()` before and after. How fast is a round trip when the client and server are on the same machine, compared to what you'd expect over a real network?

In [9]:
# Your code here


## Resources

- [MDN: HTTP request methods](https://developer.mozilla.org/en-US/docs/Web/HTTP/Methods) - GET, POST, and the others, explained in more depth.
- [MDN: HTTP response status codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Status) - the full list, beyond the three used here.
- [Flask Quickstart](https://flask.palletsprojects.com/en/stable/quickstart/) - the web framework this notebook builds on.
- [`requests` quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) - the client library used above.